# Expérience 1

Cette expérience MLflow teste une version plus robuste du dataset historique en colonnes. L'objectif est simple : comparer `RandomForest`, `XGBoost` et `XGBoost Random Forest` après avoir corrigé les principales sources de surapprentissage identifiées dans les itérations précédentes.


## Choix retenus contre l'overfitting

La configuration retenue pour cette expérience applique trois garde-fous :
- `area` n'est plus utilisée comme variable explicative ; elle sert uniquement à construire le split groupé ;
- l'historique de rendement est limité aux `3` années les plus récentes avant la cible ;
- l'évaluation se fait avec un `GroupShuffleSplit` par `area`, pour tester la généralisation sur des pays absents du train.

Cette version va directement à l'essentiel : construire le dataset, entraîner les modèles et comparer les résultats dans MLflow.


In [37]:
from datetime import date
from pathlib import Path
import sqlite3
import shutil

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRFRegressor, XGBRegressor

from scripts.project_config import DEFAULT_CONFIG_PATH, load_preparation_config

SEED = 42
pd.set_option('display.max_columns', None)


## Configuration

Le notebook relit la configuration centralisée du projet, produit un dataset historique large et journalise les résultats dans la base MLflow locale du projet.


In [ ]:
config = load_preparation_config(ensure_dirs=True)
MIN_YEAR = int(config['MIN_YEAR'])
CURRENT_YEAR = date.today().year
YEARS = list(range(MIN_YEAR, CURRENT_YEAR + 1))

YIELD_PATH = config['YIELD_PATH']
RAINFALL_PATH = config['RAINFALL_PATH']
PESTICIDES_PATH = config['PESTICIDES_PATH']
TEMP_PATH = config['TEMP_PATH']
ARTIFACTS_DIR = config['ARTIFACTS_DIR']

EXPERIENCE_DIR = ARTIFACTS_DIR / 'experiments' / 'experience_1'
EXPERIENCE_DIR.mkdir(parents=True, exist_ok=True)
EXPERIENCE_DATASET_PATH = EXPERIENCE_DIR / 'dataset_consolide_historique_colonnes.csv'
SOURCE_OVERVIEW_PATH = EXPERIENCE_DIR / 'source_overview.csv'
SOURCE_QUALITY_PATH = EXPERIENCE_DIR / 'source_quality.csv'
EXPERIENCE_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_1_summary.csv'
MISSING_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_1_missing_summary.csv'
MODEL_RESULTS_PATH = EXPERIENCE_DIR / 'model_results.csv'

MLFLOW_DB_PATH = ARTIFACTS_DIR / 'mlflow.db'
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.resolve()}"
MLFLOW_EXPERIMENT_NAME = 'ocr_projet12_experiences_donnees'
MLFLOW_ARTIFACTS_DIR = ARTIFACTS_DIR / 'mlruns'
MLFLOW_EXPERIMENT_ARTIFACT_DIR = MLFLOW_ARTIFACTS_DIR / MLFLOW_EXPERIMENT_NAME
MLFLOW_EXPERIMENT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Configuration chargée depuis : {DEFAULT_CONFIG_PATH.resolve()}')
print(f'Fenêtre historique : {MIN_YEAR}-{CURRENT_YEAR}')
print(f'Dossier expérience : {EXPERIENCE_DIR.resolve()}')
print(f'Tracking MLflow : {MLFLOW_TRACKING_URI}')


## Chargement et harmonisation des sources annuelles

On repart des quatre fichiers annuels utilisés pour la consolidation. Les clés sont harmonisées en `area`, `crop`, `year`, puis les doublons annuels sont traités avant la projection en colonnes.


In [39]:
yield_source = pd.read_csv(YIELD_PATH)
rainfall_source = pd.read_csv(RAINFALL_PATH, na_values=['..'])
pesticides_source = pd.read_csv(PESTICIDES_PATH)
temp_source = pd.read_csv(TEMP_PATH)

source_overview = pd.DataFrame(
    [
        {'fichier': 'yield.csv', 'lignes': yield_source.shape[0], 'colonnes': yield_source.shape[1], 'nan_detectes': int(yield_source.isna().sum().sum())},
        {'fichier': 'rainfall.csv', 'lignes': rainfall_source.shape[0], 'colonnes': rainfall_source.shape[1], 'nan_detectes': int(rainfall_source.isna().sum().sum())},
        {'fichier': 'pesticides.csv', 'lignes': pesticides_source.shape[0], 'colonnes': pesticides_source.shape[1], 'nan_detectes': int(pesticides_source.isna().sum().sum())},
        {'fichier': 'temp.csv', 'lignes': temp_source.shape[0], 'colonnes': temp_source.shape[1], 'nan_detectes': int(temp_source.isna().sum().sum())},
    ]
)
source_overview.to_csv(SOURCE_OVERVIEW_PATH, index=False)

yield_clean = (
    yield_source.loc[:, ['Area', 'Item', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Item': 'crop', 'Year': 'year', 'Value': 'target_yield_t_ha'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        crop=lambda df: df['crop'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        target_yield_t_ha=lambda df: pd.to_numeric(df['target_yield_t_ha'], errors='coerce') / 10000,
    )
    .dropna(subset=['area', 'crop', 'year'])
)
yield_clean = yield_clean.loc[yield_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
yield_clean['year'] = yield_clean['year'].astype(int)
TARGET_YEAR = int(yield_clean['year'].max())
FEATURE_YEARS = [year for year in YEARS if year < TARGET_YEAR]
SELECTED_YIELD_YEARS = FEATURE_YEARS[-3:]

rainfall_clean = (
    rainfall_source.loc[:, [' Area', 'Year', 'average_rain_fall_mm_per_year']]
    .rename(columns={' Area': 'area', 'Year': 'year'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        average_rain_fall_mm_per_year=lambda df: pd.to_numeric(df['average_rain_fall_mm_per_year'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
rainfall_clean = rainfall_clean.loc[rainfall_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
rainfall_clean['year'] = rainfall_clean['year'].astype(int)
rainfall_clean = rainfall_clean.drop_duplicates(subset=['area', 'year'], keep='first')

pesticides_clean = (
    pesticides_source.loc[:, ['Area', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Year': 'year', 'Value': 'pesticides_tonnes'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        pesticides_tonnes=lambda df: pd.to_numeric(df['pesticides_tonnes'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
pesticides_clean = pesticides_clean.loc[pesticides_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
pesticides_clean['year'] = pesticides_clean['year'].astype(int)
pesticides_clean = pesticides_clean.drop_duplicates(subset=['area', 'year'], keep='first')

temp_clean = (
    temp_source.loc[:, ['year', 'country', 'avg_temp']]
    .rename(columns={'country': 'area'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        avg_temp=lambda df: pd.to_numeric(df['avg_temp'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
temp_clean = temp_clean.loc[temp_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
temp_clean['year'] = temp_clean['year'].astype(int)
temp_clean = temp_clean.groupby(['area', 'year'], as_index=False)['avg_temp'].mean()

source_quality = pd.DataFrame(
    [
        {'table': 'yield_clean', 'clé': 'area + crop + year', 'doublons_sur_cle': int(yield_clean.duplicated(subset=['area', 'crop', 'year']).sum()), 'nan_totaux': int(yield_clean.isna().sum().sum())},
        {'table': 'rainfall_clean', 'clé': 'area + year', 'doublons_sur_cle': int(rainfall_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(rainfall_clean.isna().sum().sum())},
        {'table': 'pesticides_clean', 'clé': 'area + year', 'doublons_sur_cle': int(pesticides_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(pesticides_clean.isna().sum().sum())},
        {'table': 'temp_clean', 'clé': 'area + year', 'doublons_sur_cle': int(temp_clean.duplicated(subset=['area', 'year']).sum()), 'nan_totaux': int(temp_clean.isna().sum().sum())},
    ]
)
source_quality.to_csv(SOURCE_QUALITY_PATH, index=False)

display(source_overview)
display(source_quality)
print(f'Année cible retenue : {TARGET_YEAR}')
print(f'Années de rendement retenues comme features : {SELECTED_YIELD_YEARS}')


,fichier,lignes,colonnes,nan_detectes
0,yield.csv,56717,12,0
1,rainfall.csv,6727,3,780
2,pesticides.csv,4349,7,0
3,temp.csv,71311,3,2547


,table,clé,doublons_sur_cle,nan_totaux
0,yield_clean,area + crop + year,0,0
1,rainfall_clean,area + year,0,677
2,pesticides_clean,area + year,0,0
3,temp_clean,area + year,0,0


Année cible retenue : 2016
Années de rendement retenues comme features : [2013, 2014, 2015]


## Dataset historique en colonnes

Chaque variable annuelle est projetée en colonnes. La base finale contient une ligne par couple `area + crop`. La cible de modélisation est le rendement de `2016`, prédit à partir des rendements `2013-2015` et de l'historique climatique disponible.


In [40]:
def pivot_history(df: pd.DataFrame, index_cols: list[str], value_col: str, years: list[int]) -> pd.DataFrame:
    wide = df.pivot_table(index=index_cols, columns='year', values=value_col, aggfunc='first')
    wide = wide.reindex(columns=years)
    wide.columns = [f'{value_col}_{int(year)}' for year in wide.columns]
    return wide.reset_index()

base_keys = (
    yield_clean[['area', 'crop']]
    .drop_duplicates()
    .sort_values(['area', 'crop'])
    .reset_index(drop=True)
)

yield_history_wide = base_keys.merge(
    pivot_history(yield_clean, ['area', 'crop'], 'target_yield_t_ha', YEARS),
    on=['area', 'crop'],
    how='left',
    validate='1:1',
)
rainfall_history_wide = pivot_history(rainfall_clean, ['area'], 'average_rain_fall_mm_per_year', YEARS)
pesticides_history_wide = pivot_history(pesticides_clean, ['area'], 'pesticides_tonnes', YEARS)
temp_history_wide = pivot_history(temp_clean, ['area'], 'avg_temp', YEARS)

experience_1_dataset = (
    yield_history_wide
    .merge(rainfall_history_wide, on='area', how='left', validate='m:1')
    .merge(pesticides_history_wide, on='area', how='left', validate='m:1')
    .merge(temp_history_wide, on='area', how='left', validate='m:1')
    .sort_values(['area', 'crop'])
    .reset_index(drop=True)
)

missing_summary = (
    experience_1_dataset.isna()
    .sum()
    .rename('nb_nan')
    .reset_index()
    .rename(columns={'index': 'variable'})
)
missing_summary['part_nan_pct'] = (missing_summary['nb_nan'] / len(experience_1_dataset) * 100).round(2)
missing_summary.to_csv(MISSING_SUMMARY_PATH, index=False)

experience_summary = pd.DataFrame(
    {
        'indicateur': [
            'nb_lignes',
            'nb_colonnes',
            'annee_cible_modele',
            'part_nan_globale_pct',
            'colonnes_cible_historiques',
            'colonnes_pluie_historiques',
            'colonnes_pesticides_historiques',
            'colonnes_temperature_historiques',
        ],
        'valeur': [
            int(experience_1_dataset.shape[0]),
            int(experience_1_dataset.shape[1]),
            TARGET_YEAR,
            round(experience_1_dataset.isna().mean().mean() * 100, 2),
            len([c for c in experience_1_dataset.columns if c.startswith('target_yield_t_ha_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('average_rain_fall_mm_per_year_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('pesticides_tonnes_')]),
            len([c for c in experience_1_dataset.columns if c.startswith('avg_temp_')]),
        ],
    }
)
experience_summary.to_csv(EXPERIENCE_SUMMARY_PATH, index=False)
experience_1_dataset.to_csv(EXPERIENCE_DATASET_PATH, index=False)

display(experience_summary)
display(missing_summary.head(15))
print(f'Dataset expérience 1 sauvegardé : {EXPERIENCE_DATASET_PATH.resolve()}')


,indicateur,valeur
0,nb_lignes,1168.00
1,nb_colonnes,150.00
2,annee_cible_modele,2016.00
3,part_nan_globale_pct,44.51
4,colonnes_cible_historiques,37.00
5,colonnes_pluie_historiques,37.00
6,colonnes_pesticides_historiques,37.00
7,colonnes_temperature_historiques,37.00


,variable,nb_nan,part_nan_pct
0,area,0,0.00
1,crop,0,0.00
2,target_yield_t_ha_1990,179,15.33
3,target_yield_t_ha_1991,181,15.50
4,target_yield_t_ha_1992,102,8.73
5,target_yield_t_ha_1993,93,7.96
6,target_yield_t_ha_1994,89,7.62
7,target_yield_t_ha_1995,83,7.11
8,target_yield_t_ha_1996,85,7.28
9,target_yield_t_ha_1997,81,6.93


Dataset expérience 1 sauvegardé : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1/dataset_consolide_historique_colonnes.csv


## Jeu de modélisation retenu

Ici, `area` est retirée des features mais conservée pour le `GroupShuffleSplit`. Le one-hot encoding ne porte plus que sur `crop`, ce qui réduit fortement la taille de la matrice encodée. Les colonnes numériques entièrement vides côté train sont supprimées avant entraînement.


In [41]:
target_col = f'target_yield_t_ha_{TARGET_YEAR}'

full_feature_pool_cols = ['area', 'crop']
full_feature_pool_cols += [f'target_yield_t_ha_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'average_rain_fall_mm_per_year_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'pesticides_tonnes_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols += [f'avg_temp_{year}' for year in FEATURE_YEARS]
full_feature_pool_cols = [col for col in full_feature_pool_cols if col in experience_1_dataset.columns]

feature_cols = ['crop']
feature_cols += [f'target_yield_t_ha_{year}' for year in SELECTED_YIELD_YEARS]
feature_cols += [f'average_rain_fall_mm_per_year_{year}' for year in FEATURE_YEARS]
feature_cols += [f'pesticides_tonnes_{year}' for year in FEATURE_YEARS]
feature_cols += [f'avg_temp_{year}' for year in FEATURE_YEARS]
feature_cols = [col for col in feature_cols if col in experience_1_dataset.columns]

base_columns = ['area'] + [col for col in feature_cols if col != 'area'] + [target_col]
base_columns = list(dict.fromkeys(base_columns))
model_df = experience_1_dataset[base_columns].copy()
model_df = model_df.dropna(subset=[target_col]).reset_index(drop=True)

categorical_features = ['crop']
numeric_features = [col for col in feature_cols if col not in categorical_features]

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()
groups = model_df['area'].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

train_empty_numeric_features = [col for col in numeric_features if not X_train[col].notna().any()]
numeric_features = [col for col in numeric_features if col not in train_empty_numeric_features]
feature_cols = categorical_features + numeric_features
X_train = X_train[feature_cols].copy()
X_test = X_test[feature_cols].copy()

shared_areas = sorted(set(groups_train) & set(groups_test))

modeling_overview = pd.DataFrame(
    {
        'indicateur': [
            'train_rows',
            'test_rows',
            'train_areas',
            'test_areas',
            'shared_areas_between_train_and_test',
            'selected_feature_count',
            'selected_yield_year_count',
            'numeric_feature_count',
            'categorical_feature_count',
            'area_used_as_feature',
            'area_used_as_group_only',
            'dropped_train_empty_numeric_features',
        ],
        'valeur': [
            X_train.shape[0],
            X_test.shape[0],
            groups_train.nunique(),
            groups_test.nunique(),
            len(shared_areas),
            len(feature_cols),
            len(SELECTED_YIELD_YEARS),
            len(numeric_features),
            len(categorical_features),
            0,
            1,
            len(train_empty_numeric_features),
        ],
    }
)
display(modeling_overview)
display(pd.DataFrame({'selected_yield_year': SELECTED_YIELD_YEARS}))
if train_empty_numeric_features:
    display(pd.DataFrame({'dropped_train_empty_numeric_feature': train_empty_numeric_features}))
print(f'Cible : {target_col}')


,indicateur,valeur
0,train_rows,903
1,test_rows,208
2,train_areas,163
3,test_areas,41
4,shared_areas_between_train_and_test,0
5,selected_feature_count,79
6,selected_yield_year_count,3
7,numeric_feature_count,78
8,categorical_feature_count,1
9,area_used_as_feature,0


,selected_yield_year
0,2013
1,2014
2,2015


,dropped_train_empty_numeric_feature
0,average_rain_fall_mm_per_year_2003
1,avg_temp_2014
2,avg_temp_2015


Cible : target_yield_t_ha_2016


## Encodage et dimension finale

Le `one-hot encoding` ne porte plus que sur `crop`. Cette réduction de cardinalité est un des choix centraux faits pour limiter la mémorisation du train.


In [42]:
def build_preprocessor() -> ColumnTransformer:
    numeric_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ],
        sparse_threshold=0.0,
    )

probe_preprocessor = build_preprocessor()
X_train_prepared = probe_preprocessor.fit_transform(X_train)
onehot_encoder = probe_preprocessor.named_transformers_['cat'].named_steps['onehot']
onehot_modalities = int(sum(len(categories) for categories in onehot_encoder.categories_))
encoded_feature_count = int(X_train_prepared.shape[1])

ohe_overview = pd.DataFrame(
    {
        'indicateur': ['raw_feature_count', 'onehot_modalities', 'encoded_feature_count'],
        'valeur': [len(feature_cols), onehot_modalities, encoded_feature_count],
    }
)
display(ohe_overview)


,indicateur,valeur
0,raw_feature_count,79
1,onehot_modalities,10
2,encoded_feature_count,88


## Entraînement et suivi MLflow

Les trois modèles sont comparés sur la même base et la même préparation. Les métriques `train` et `test` sont loggées pour garder de la visibilité sur le surapprentissage, même si l'expérience est maintenant recentrée sur une configuration déjà corrigée.


In [43]:
candidate_models = {
    'random_forest': RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    ),
    'xgboost': XGBRegressor(
        objective='reg:squarederror',
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    ),
    'xgboost_random_forest': XGBRFRegressor(
        objective='reg:squarederror',
        n_estimators=400,
        max_depth=8,
        learning_rate=1.0,
        subsample=0.8,
        colsample_bynode=0.8,
        reg_lambda=1.0,
        tree_method='hist',
        random_state=SEED,
        n_jobs=-1,
    ),
}

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
while mlflow.active_run() is not None:
    mlflow.end_run()

tracking_db_path = Path(MLFLOW_TRACKING_URI.removeprefix('sqlite:///'))
experiment_artifact_uri = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve().as_uri()
tracking_db_path.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if tracking_db_path.exists():
    connection = sqlite3.connect(tracking_db_path)
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='experiments'")
    if cursor.fetchone() is not None:
        cursor.execute(
            'SELECT experiment_id, name, artifact_location FROM experiments WHERE name = ?',
            (MLFLOW_EXPERIMENT_NAME,),
        )
        existing_row = cursor.fetchone()
        if existing_row is not None:
            experiment_id, _, current_artifact_location = existing_row
            current_artifact_dir = Path(str(current_artifact_location).removeprefix('file://')).resolve()
            target_artifact_dir = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve()
            if current_artifact_dir.exists() and current_artifact_dir != target_artifact_dir:
                for child in current_artifact_dir.iterdir():
                    destination = target_artifact_dir / child.name
                    if not destination.exists():
                        shutil.move(str(child), str(destination))
                if current_artifact_dir.exists() and current_artifact_dir.is_dir() and not any(current_artifact_dir.iterdir()):
                    current_artifact_dir.rmdir()
            cursor.execute(
                'UPDATE experiments SET artifact_location = ? WHERE experiment_id = ?',
                (experiment_artifact_uri, experiment_id),
            )
            cursor.execute(
                'UPDATE runs SET artifact_uri = REPLACE(artifact_uri, ?, ?) WHERE experiment_id = ? AND artifact_uri LIKE ?',
                (
                    str(current_artifact_dir),
                    str(target_artifact_dir),
                    experiment_id,
                    f'{current_artifact_dir}%',
                ),
            )
            connection.commit()
    connection.close()

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    client.create_experiment(MLFLOW_EXPERIMENT_NAME, artifact_location=experiment_artifact_uri)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

results = []
for model_name, estimator in candidate_models.items():
    pipeline = Pipeline(
        steps=[
            ('preprocessor', build_preprocessor()),
            ('regressor', estimator),
        ]
    )

    with mlflow.start_run(run_name=f'experience_1__{model_name}') as run:
        pipeline.fit(X_train, y_train)

        train_pred = pipeline.predict(X_train)
        test_pred = pipeline.predict(X_test)

        train_mae = float(mean_absolute_error(y_train, train_pred))
        train_rmse = float(np.sqrt(mean_squared_error(y_train, train_pred)))
        train_r2 = float(r2_score(y_train, train_pred))
        test_mae = float(mean_absolute_error(y_test, test_pred))
        test_rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
        test_r2 = float(r2_score(y_test, test_pred))
        overfit_gap_rmse = float(test_rmse - train_rmse)
        overfit_ratio_rmse = float(test_rmse / train_rmse) if train_rmse > 0 else np.nan

        mlflow.log_param('experience_name', 'experience_1')
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('target_col', target_col)
        mlflow.log_param('target_year', TARGET_YEAR)
        mlflow.log_param('split_strategy', 'GroupShuffleSplit(area)')
        mlflow.log_param('area_used_as_feature', False)
        mlflow.log_param('selected_yield_year_count', len(SELECTED_YIELD_YEARS))
        mlflow.log_param('selected_yield_year_start', int(SELECTED_YIELD_YEARS[0]))
        mlflow.log_param('selected_yield_year_end', int(SELECTED_YIELD_YEARS[-1]))
        mlflow.log_param('encoded_feature_count', encoded_feature_count)
        mlflow.log_param('onehot_modalities', onehot_modalities)

        mlflow.log_metric('train_mae', train_mae)
        mlflow.log_metric('train_rmse', train_rmse)
        mlflow.log_metric('train_r2', train_r2)
        mlflow.log_metric('test_mae', test_mae)
        mlflow.log_metric('test_rmse', test_rmse)
        mlflow.log_metric('test_r2', test_r2)
        mlflow.log_metric('overfit_gap_rmse', overfit_gap_rmse)
        mlflow.log_metric('overfit_ratio_rmse', overfit_ratio_rmse)

        mlflow.log_artifact(str(EXPERIENCE_SUMMARY_PATH))
        mlflow.log_artifact(str(MISSING_SUMMARY_PATH))
        mlflow.log_artifact(str(EXPERIENCE_DATASET_PATH))
        mlflow.sklearn.log_model(pipeline, name='model')

        results.append(
            {
                'model': model_name,
                'train_mae': train_mae,
                'train_rmse': train_rmse,
                'train_r2': train_r2,
                'test_mae': test_mae,
                'test_rmse': test_rmse,
                'test_r2': test_r2,
                'overfit_gap_rmse': overfit_gap_rmse,
                'overfit_ratio_rmse': overfit_ratio_rmse,
                'run_id': run.info.run_id,
            }
        )

results_df = pd.DataFrame(results).sort_values('test_rmse').reset_index(drop=True)
results_df.to_csv(MODEL_RESULTS_PATH, index=False)

with mlflow.start_run(run_name='experience_1__summary'):
    mlflow.log_param('experience_name', 'experience_1')
    mlflow.log_param('models_tested', ','.join(candidate_models.keys()))
    mlflow.log_param('selected_feature_strategy', 'no_area_plus_recent_3_yield_years')
    mlflow.log_metric('best_test_rmse', float(results_df.loc[0, 'test_rmse']))
    mlflow.log_metric('best_test_r2', float(results_df.loc[0, 'test_r2']))
    mlflow.log_artifact(str(MODEL_RESULTS_PATH))

print(f'Résultats sauvegardés : {MODEL_RESULTS_PATH.resolve()}')


2026/04/18 18:19:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 18:20:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/18 18:20:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Résultats sauvegardés : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_1/model_results.csv


## Lecture finale

Cette expérience retient une configuration volontairement plus sobre. Elle garde de la visibilité sur l'overfitting via les métriques `train/test`, mais elle évite les blocs de diagnostic exploratoire et va directement au résultat comparatif.


In [44]:
display(results_df)
print('Configuration retenue : area retirée des features, historique de rendement réduit à 3 années récentes.')
print(f'Années de rendement retenues : {SELECTED_YIELD_YEARS}')
print(f"Meilleur modèle : {results_df.loc[0, 'model']}")
print(f"Run MLflow du meilleur modèle : {results_df.loc[0, 'run_id']}")


,model,train_mae,train_rmse,train_r2,test_mae,test_rmse,test_r2,overfit_gap_rmse,overfit_ratio_rmse,run_id
0,xgboost_random_forest,0.566441,1.298934,0.974820,0.891290,2.193577,0.939677,0.894643,1.688751,7636973b51004c4b8cc495aa85847f1a
1,random_forest,0.293445,0.850912,0.989194,0.837602,2.238686,0.937171,1.387774,2.630924,0667d99fc19147b8accbbf9d03ad350f
2,xgboost,0.112001,0.163362,0.999602,0.925761,2.318342,0.932620,2.154980,14.191420,357fb865d2394ec996389812b176822f


Configuration retenue : area retirée des features, historique de rendement réduit à 3 années récentes.
Années de rendement retenues : [2013, 2014, 2015]
Meilleur modèle : xgboost_random_forest
Run MLflow du meilleur modèle : 7636973b51004c4b8cc495aa85847f1a
